In [ ]:
#Local Beam Search
import heapq
graph = {
    'S': [('A', 3), ('B', 6), ('C', 5)],
    'A': [('D', 9), ('E', 8)],
    'B': [('F', 12),
    ('G', 14)],
    'C': [('H', 7)],
    'H': [('I', 5),
    ('J', 6)],
    'I': [('K', 1),
    ('L', 10), ('M', 2)],
    'D': [],'E': [],
    'F': [],'G': [],
    'J': [],'K': [],
    'L': [],'M': []
}

def beam_search(start,goal,beam_width=2):
    beam = [(0, [start])]   #total cost and path
    while beam:
        candidates = []
        #expand each node in beam
        for cost,path in beam:
            current_node = path[-1] #evaluate the last node in path
            if current_node == goal:
                return path,cost
            
            #generate successors
            for neighbour,edge_cost in graph.get(current_node,[]):   #returns the path and mapped cost
                new_cost = cost+edge_cost   #adding bcz its cumulative
                new_path = path + [neighbour]
                candidates.append((new_cost,new_path))

            #select top k paths based on cumulative cost
            beam = heapq.nsmallest(beam_width,candidates,key=lambda x:x[0])

    return None, float('inf')

start = 'S'
goal = 'L'
k=3
path, cost = beam_search(start,goal,k)

if path:
    print(f"Path Found: {{{path}}} with cost={cost}")


Path Found: {['S', 'C', 'H', 'I', 'L']} with cost=27


In [ ]:
#Hill Climbing (N-queens)
import random
#huerestic to calculate conflicting queens
def calc_conflicts(state):
    print(state)
    conflicts=0
    n=len(state)    #state is 1D bcz index represent column
    for i in range(n):
        for j in range(i+1,n):
            if state[i]==state[j] or abs(state[i]-state[j])==abs(i-j):
                conflicts+=1

    return conflicts

#generate neighbours by moving 1 Queen at a time
def get_neighbours(state):
    neighbours = []
    n = len(state)
    for col in range(n):
        for row in range(n):
            if row != state[col]:   #non-attacking row and col
                new_state = list(state)     #list makes a copy
                new_state[col] = row
                neighbours.append(new_state)
    return neighbours

#Simple Hill Climbing
def simple_hill_climbing(n):
    current_state = []
    for _ in range(n):
        random_row = random.randint(0,n-1)
        current_state.append(random_row)
    current_conflicts = calc_conflicts(current_state)

    while True:
        neighbours = get_neighbours(current_state)
        next_state = None
        next_conflicts = current_conflicts

            #find first better neighbour
        for neighbour in neighbours:
            neighbour_conflicts = calc_conflicts(neighbour)
            if neighbour_conflicts < next_conflicts:
                next_state = neighbour
                next_conflicts = neighbour_conflicts
                break   #moved to better neighbour

        #if no better neighbour found, return curren state
        if next_conflicts>=current_conflicts:
            break

        #move
        current_state=next_state
        current_conflicts=next_conflicts
    return current_state,current_conflicts
    

solution,conflicts = simple_hill_climbing(4)
if conflicts == 0:
    print("Solution found for N-Queen:-\n")
    print(solution)
else:
    print(f"Couldn't find solution. Stuck at state with {conflicts} conflicts")
    print(solution)            

[2, 0, 3, 0]
[0, 0, 3, 0]
[1, 0, 3, 0]
[3, 0, 3, 0]
[2, 1, 3, 0]
[2, 2, 3, 0]
[2, 3, 3, 0]
[2, 0, 0, 0]
[2, 0, 1, 0]
[2, 0, 2, 0]
[2, 0, 3, 1]
[0, 0, 3, 1]
[1, 0, 3, 1]
[3, 0, 3, 1]
[2, 1, 3, 1]
[2, 2, 3, 1]
[2, 3, 3, 1]
[2, 0, 0, 1]
[2, 0, 1, 1]
[2, 0, 2, 1]
[2, 0, 3, 0]
[2, 0, 3, 2]
[2, 0, 3, 3]
Solution found for N-Queen:-

[2, 0, 3, 1]


In [25]:
#Genetic Algorithm (N-queen)
import random
n=8
#fitness function - counts non attacking pairs of queen
def calculate_fitness(individual):
    non_attacking_pairs=0
    total_pairs = n*(n-1)//2    #formula for max possible non attacking pair
    for i in range(n):
        for j in range(i+1,n):
            if individual[i]!=individual[j] and abs(individual[i]-individual[j])!=abs(i-j):
                non_attacking_pairs += 1
    
    return non_attacking_pairs/total_pairs

def create_random_individual():
    return random.sample(range(n),n)

def select_parents(population, fitness_scores):
    sorted_population = [board for _, board in sorted(zip(fitness_scores,
population), reverse=True)]
    return sorted_population[:len(population)//2]

def crossover(parent1,parent2):
    point = random.randint(1,n-2)
    child = parent1[:point]+parent2[point:]
    
    #if duplicate values are found (means 2 queens in same row), remove them and replace with missing number
    missing = list(set(range(n)) - set(child))
    for i in range(len(child)):
        if child.count(child[i]) > 1:
            child[i] = missing.pop()
    return child

def mutation(individual):
    idx1, idx2 = random.sample(range(n),2)
    individual[idx1], individual[idx2] = individual[idx2], individual[idx1]
    return individual


def genetic_algorithm():
    populationSize=10
    generation=0
    population = [create_random_individual() for _ in range(populationSize)]

    while True:
        generation+=1
        fitness_scores = [calculate_fitness(ind) for ind in population]
        best_fitness = max(fitness_scores)
        best_individual = population[fitness_scores.index(best_fitness)]

        print(f"Generation: {generation} | Fitness: {best_fitness} | State: {best_individual}")
        if best_fitness==1.0:
            print("\nSolution Found")
            return best_individual

        #select top 50% parents and mutate reproduce
        parents = select_parents(population, fitness_scores)
        new_population = []
        while len(new_population) < populationSize:
            p1, p2 = random.sample(parents,2)
            child = crossover(p1,p2)

            #10% mutation chance
            if random.random() < 0.1:
                child = mutation(child)
            new_population.append(child)

        population = new_population

solution = genetic_algorithm()
print(f"Final Board State: {solution}")        
        


Generation: 1 | Fitness: 0.8928571428571429 | State: [6, 4, 7, 1, 0, 3, 5, 2]
Generation: 2 | Fitness: 0.9285714285714286 | State: [0, 4, 7, 5, 2, 1, 3, 6]
Generation: 3 | Fitness: 0.8928571428571429 | State: [1, 6, 7, 0, 2, 4, 5, 3]
Generation: 4 | Fitness: 0.8928571428571429 | State: [6, 1, 7, 0, 2, 4, 5, 3]
Generation: 5 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 6 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 7 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 8 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 9 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 10 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 11 | Fitness: 0.9285714285714286 | State: [6, 1, 2, 5, 7, 4, 3, 0]
Generation: 12 | Fitness: 0.9285714285714286 | State: [6, 4, 2, 5, 7, 1, 3, 0]
Generation: 13 | Fitness: 0.9285714285714286 | State: [6, 4, 

In [ ]:
#Genetic Algorithm - Duty Scheduling
"""A hospital needs to create a weekly shift schedule for its staff, ensuring that there are
enough staff members on duty during peak hours, while also giving each staff member
proper rest. """

#here fitness function will be evaluated by penalty score in each shift which is calculated as:
#1. shift coverage - if each shift has required staff members and each missing means penalty of 10
#2. consecutive shifts - if an employee is assigned 2 consecutive shifts or more than 7 shifts a week, add penalty of 5

import random
#we have 7 days and 3 shifts a day
numStaff=5, numShifts=21
maxShiftPerStaff=7, requiredShiftPerStaff=2     #assumed (like business rule)
populationSize=10,  mutation_rate=0.1, maxGens=1000

def evaluateFitness(schedule):
    penalty=0
    #for understaffed:
    for i in range(numShifts):
        staffOnDuty=0
        for j in range(numStaff):
            if schedule[i][j]==1:
                staffOnDuty+=1

        if staffOnDuty < requiredShiftPerStaff:
            penalty+=10
    
    #staff workload management:
    for i in range(numStaff):
        shiftsWorked=0
        for j in range(numShifts):
            if schedule[i][j]==1:
                shiftsWorked+=1
                if j<numShifts-1 and schedule[i][j+1]==1:
                    penalty+=5

        if shiftsWorked>maxShiftPerStaff:
            penalty += (shiftsWorked-maxShiftPerStaff)*5    #penalty of 5 per extra shift of employee
    
    return penalty

schedule = [
    [1,0,0,0,1,0,0]
    [1,1,1,0,0,1,1]
    [0,0,1,1,0,0,1]
    [0,1,0,1,0,1,0]
    [1,0,0,1,0,1,1]
]